In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.LM(
    params=model.parameters(),
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=False,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="solve",
    batch_size=100,
    # batch_size=None,
)

all_loss = {}
for epoch in range(500):
    print("epoch: ", epoch, end="")
    all_loss[epoch + 1] = 0
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step(b_x, b_y, loss_fn)

        all_loss[epoch + 1] += loss
    all_loss[epoch + 1] /= len(data_loader)
    print(", loss: {}".format(all_loss[epoch + 1].cpu().detach().numpy().item()))

epoch:  0, loss: 0.32538744807243347
epoch:  1, loss: 0.27499163150787354
epoch:  2, loss: 0.21432100236415863
epoch:  3, loss: 0.1475532203912735
epoch:  4, loss: 0.0870857909321785
epoch:  5, loss: 0.04554259404540062
epoch:  6, loss: 0.03194585070014
epoch:  7, loss: 0.02193787321448326
epoch:  8, loss: 0.020549843087792397
epoch:  9, loss: 0.019035598263144493
epoch:  10, loss: 0.01748179830610752
epoch:  11, loss: 0.0161043182015419
epoch:  12, loss: 0.01557115651667118
epoch:  13, loss: 0.014885603450238705
epoch:  14, loss: 0.014463085681200027
epoch:  15, loss: 0.014188175089657307
epoch:  16, loss: 0.013760365545749664
epoch:  17, loss: 0.013215739279985428
epoch:  18, loss: 0.012926540337502956
epoch:  19, loss: 0.012756376527249813
epoch:  20, loss: 0.012467251159250736
epoch:  21, loss: 0.012140678241848946
epoch:  22, loss: 0.01197450514882803
epoch:  23, loss: 0.011879836209118366
epoch:  24, loss: 0.011696594767272472
epoch:  25, loss: 0.01149438414722681
epoch:  26, los

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9337551236392516
Test metrics:  R2 = 0.9141507074811653


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.LM(
    params=model.parameters(),
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=True,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="pinv",
    batch_size=100,
    # batch_size=None,
)

all_loss = {}
for epoch in range(500):
    print("epoch: ", epoch, end="")
    all_loss[epoch + 1] = 0
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step(b_x, b_y, loss_fn)

        all_loss[epoch + 1] += loss
    all_loss[epoch + 1] /= len(data_loader)
    print(", loss: {}".format(all_loss[epoch + 1].cpu().detach().numpy().item()))

epoch:  0, loss: 0.612861692905426
epoch:  1, loss: 0.4568637013435364
epoch:  2, loss: 0.0401274599134922
epoch:  3, loss: 0.029215607792139053
epoch:  4, loss: 0.02433541975915432
epoch:  5, loss: 0.019596727564930916
epoch:  6, loss: 0.016591835767030716
epoch:  7, loss: 0.015542183071374893
epoch:  8, loss: 0.013912403956055641
epoch:  9, loss: 0.01277364045381546
epoch:  10, loss: 0.01216751616448164
epoch:  11, loss: 0.011813279241323471
epoch:  12, loss: 0.010938966646790504
epoch:  13, loss: 0.010237831622362137
epoch:  14, loss: 0.009968125261366367
epoch:  15, loss: 0.009832285344600677
epoch:  16, loss: 0.009374742396175861
epoch:  17, loss: 0.00911962240934372
epoch:  18, loss: 0.009051204659044743
epoch:  19, loss: 0.009036500006914139
epoch:  20, loss: 0.00887478981167078
epoch:  21, loss: 0.008839827962219715
epoch:  22, loss: 0.008829601109027863
epoch:  23, loss: 0.00880475901067257
epoch:  24, loss: 0.008790108375251293
epoch:  25, loss: 0.008786662481725216
epoch:  2

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6840875923929568
Test metrics:  R2 = 0.7279515275961823
